# 03 - Grad-CAM Explainability (XAI)

Mục tiêu: Tạo Grad-CAM heatmap cho model classification đã training từ notebook 02.
Chạy cho 100 ảnh mỗi lớp ở tập train. Visualization sample ảnh mỗi lớp.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

# Auto-detect project root
notebook_dir = os.path.abspath('')
proj_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
sys.path.insert(0, proj_root)
sys.path.insert(0, os.path.join(proj_root, 'utils'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import cv2
import glob
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from torchvision import transforms

from utils.models import EfficientNetClassifier
from utils.gradcam import GradCAMGenerator, find_efficientnet_target_layer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('proj_root:', proj_root)

device: cpu
proj_root: E:\Master\thesis_durian


## 1) Load train data

In [ ]:
# Đường dẫn tới tập train
train_dir = os.path.join(proj_root, 'notebooks', 'data', 'raw', 'train')
if not os.path.exists(train_dir):
    train_dir = 'data/raw/train'

print('Train dir:', train_dir)
print('Exists:', os.path.exists(train_dir))

# Lấy danh sách class
class_dirs = sorted([d for d in Path(train_dir).iterdir() if d.is_dir()])
class_names = [d.name for d in class_dirs]
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}

print(f'\nClasses ({len(class_names)}):', class_names)

# Lấy 100 ảnh mỗi lớp
SAMPLES_PER_CLASS = 100
selected_samples = []  # list of (img_path, class_idx, class_name)

for cls_dir in class_dirs:
    cls_name = cls_dir.name
    cls_idx = class_to_idx[cls_name]
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')) + list(cls_dir.glob('*.jpeg'))
    # imgs = sorted(imgs)[:SAMPLES_PER_CLASS]
    for img_path in imgs:
        selected_samples.append((str(img_path), cls_idx, cls_name))
    print(f'  {cls_name}: {len(imgs)} ảnh')

print(f'\nTổng: {len(selected_samples)} ảnh')

Train dir: E:\Master\thesis_durian\notebooks\data\raw\train
Exists: True

Classes (5): ['ALGAL_LEAF_SPOT', 'ALLOCARIDARA_ATTACK', 'HEALTHY_LEAF', 'LEAF_BLIGHT', 'PHOMOPSIS_LEAF_SPOT']
  ALGAL_LEAF_SPOT: 100 ảnh
  ALLOCARIDARA_ATTACK: 100 ảnh
  HEALTHY_LEAF: 100 ảnh
  LEAF_BLIGHT: 100 ảnh
  PHOMOPSIS_LEAF_SPOT: 100 ảnh

Tổng: 500 ảnh


## 2) Load best classification model

In [ ]:
num_classes = len(class_names)
model = EfficientNetClassifier(num_classes=num_classes, backbone='efficientnet_b0', pretrained=False)

# Tìm checkpoint
ckpt_patterns = [
    os.path.join(proj_root, 'models', 'classification', 'checkpoints', '*.pth'),
    'models/classification/checkpoints/*.pth',
]
candidates = []
for pat in ckpt_patterns:
    candidates.extend(glob.glob(pat))

if not candidates:
    raise FileNotFoundError('Không tìm thấy checkpoint classification. Chạy notebook 02 trước.')

ckpt_path = max(candidates, key=os.path.getctime)
print('Loading checkpoint:', ckpt_path)

checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()
print('Model loaded OK')

Loading checkpoint: models/classification/checkpoints\efficientnet_baseline_best_9751.pth


Model loaded OK


## 3) Setup Grad-CAM

In [ ]:
target_layer = find_efficientnet_target_layer(model)
gradcam_gen = GradCAMGenerator(model, target_layer=target_layer, device=device.type)
print('Grad-CAM target layer:', target_layer)

# Transform cho inference
infer_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Grad-CAM target layer: base.features.8.0


## 4) Chạy Grad-CAM cho 100 ảnh mỗi lớp

In [ ]:
# Lưu kết quả theo class
results_by_class = defaultdict(list)  # class_name -> list of dicts

for img_path, cls_idx, cls_name in tqdm(selected_samples, desc='Generating GradCAM'):
    try:
        # Load ảnh gốc (RGB, 224x224)
        orig_img = Image.open(img_path).convert('RGB').resize((224, 224))
        orig_arr = np.array(orig_img)  # uint8 [H,W,3]

        # Tensor cho model
        input_tensor = infer_transform(orig_img).unsqueeze(0).to(device)

        # Grad-CAM
        heatmap, confidence, pred_class = gradcam_gen.generate(input_tensor, cls_idx)

        # Overlay
        heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(orig_arr, 0.6, heatmap_rgb, 0.4, 0)

        results_by_class[cls_name].append({
            'img_path': img_path,
            'orig_arr': orig_arr,
            'heatmap': heatmap,
            'overlay': overlay,
            'confidence': confidence,
            'pred_class': pred_class,
            'true_class': cls_idx,
        })
    except Exception as e:
        print(f'  Lỗi {img_path}: {e}')

print('\nKết quả:')
for cls_name, items in results_by_class.items():
    correct = sum(1 for x in items if x['pred_class'] == x['true_class'])
    print(f'  {cls_name}: {len(items)} ảnh, accuracy={correct/len(items):.2%}')


Generating GradCAM:   0%|          | 0/500 [00:00<?, ?it/s]


Generating GradCAM:   0%|          | 1/500 [00:00<07:29,  1.11it/s]


Generating GradCAM:   0%|          | 2/500 [00:01<05:51,  1.42it/s]


Generating GradCAM:   1%|          | 3/500 [00:02<05:17,  1.56it/s]


Generating GradCAM:   1%|          | 4/500 [00:02<04:59,  1.66it/s]

...


Generating GradCAM: 100%|█████████▉| 498/500 [05:43<00:01,  1.26it/s]


Generating GradCAM: 100%|█████████▉| 499/500 [05:44<00:00,  1.27it/s]


Generating GradCAM: 100%|██████████| 500/500 [05:45<00:00,  1.36it/s]


Generating GradCAM: 100%|██████████| 500/500 [05:45<00:00,  1.45it/s]


Kết quả:
  ALGAL_LEAF_SPOT: 100 ảnh, accuracy=100.00%
  ALLOCARIDARA_ATTACK: 100 ảnh, accuracy=100.00%
  HEALTHY_LEAF: 100 ảnh, accuracy=100.00%
  LEAF_BLIGHT: 100 ảnh, accuracy=98.00%
  PHOMOPSIS_LEAF_SPOT: 100 ảnh, accuracy=100.00%


## 5) Visualization - Sample mỗi lớp

In [ ]:
# Vẽ 3 ảnh sample mỗi lớp: Original | GradCAM Heatmap | Overlay
n_classes = len(class_names)
n_samples_show = 3  # số ảnh mỗi lớp để hiển thị

fig, axes = plt.subplots(n_classes * n_samples_show, 3, figsize=(15, 5 * n_classes * n_samples_show))
if n_classes * n_samples_show == 1:
    axes = axes.reshape(1, -1)

row = 0
for cls_name in class_names:
    items = results_by_class.get(cls_name, [])
    for s_idx in range(min(n_samples_show, len(items))):
        item = items[s_idx]
        correct_str = '✓' if item['pred_class'] == item['true_class'] else '✗'

        axes[row, 0].imshow(item['orig_arr'])
        axes[row, 0].set_title(f'{cls_name}\nOriginal', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(item['heatmap'], cmap='jet')
        axes[row, 1].set_title(f'GradCAM Heatmap\nConf={item["confidence"]:.3f} {correct_str}', fontsize=9)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(item['overlay'])
        axes[row, 2].set_title(f'Overlay\nPred={idx_to_class.get(item["pred_class"], item["pred_class"])}', fontsize=9)
        axes[row, 2].axis('off')

        row += 1

plt.suptitle('Grad-CAM Visualization - Train Set (3 samples/class)', fontsize=14, y=1.001)
plt.tight_layout()

# Lưu
out_dir = os.path.join(proj_root, 'models', 'classification', 'gradcam')
os.makedirs(out_dir, exist_ok=True)
save_path = os.path.join(out_dir, 'gradcam_train_samples.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: E:\Master\thesis_durian\models\classification\gradcam\gradcam_train_samples.png


C:\Users\pc\AppData\Local\Temp\ipykernel_13980\3307632033.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Vẽ thêm: 1 ảnh đại diện mỗi lớp (best confidence)
fig2, axes2 = plt.subplots(n_classes, 3, figsize=(15, 5 * n_classes))
if n_classes == 1:
    axes2 = axes2.reshape(1, -1)

for i, cls_name in enumerate(class_names):
    items = results_by_class.get(cls_name, [])
    if not items:
        continue
    # Chọn ảnh có confidence cao nhất
    best = max(items, key=lambda x: x['confidence'])

    axes2[i, 0].imshow(best['orig_arr'])
    axes2[i, 0].set_title(f'{cls_name}\nOriginal', fontsize=10)
    axes2[i, 0].axis('off')

    axes2[i, 1].imshow(best['heatmap'], cmap='jet')
    axes2[i, 1].set_title(f'GradCAM Heatmap\nConf={best["confidence"]:.3f}', fontsize=10)
    axes2[i, 1].axis('off')

    axes2[i, 2].imshow(best['overlay'])
    axes2[i, 2].set_title(f'Overlay (Best Conf)', fontsize=10)
    axes2[i, 2].axis('off')

plt.suptitle('Grad-CAM Best Sample per Class', fontsize=14)
plt.tight_layout()
save_path2 = os.path.join(out_dir, 'gradcam_best_per_class.png')
plt.savefig(save_path2, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path2}')

Saved: E:\Master\thesis_durian\models\classification\gradcam\gradcam_best_per_class.png


C:\Users\pc\AppData\Local\Temp\ipykernel_13980\3395245860.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6) Summary Statistics

In [ ]:
import json

summary = {}
for cls_name, items in results_by_class.items():
    correct = sum(1 for x in items if x['pred_class'] == x['true_class'])
    confs = [x['confidence'] for x in items]
    summary[cls_name] = {
        'n_samples': len(items),
        'accuracy': correct / len(items) if items else 0,
        'mean_confidence': float(np.mean(confs)) if confs else 0,
        'min_confidence': float(np.min(confs)) if confs else 0,
        'max_confidence': float(np.max(confs)) if confs else 0,
    }

print('=== GradCAM Summary (Train set, 100 ảnh/lớp) ===')
for cls_name, stats in summary.items():
    print(f"  {cls_name}: n={stats['n_samples']}, acc={stats['accuracy']:.2%}, "
          f"conf_mean={stats['mean_confidence']:.3f}")

# Lưu summary
summary_path = os.path.join(out_dir, 'gradcam_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved: {summary_path}')

=== GradCAM Summary (Train set, 100 ảnh/lớp) ===
  ALGAL_LEAF_SPOT: n=100, acc=100.00%, conf_mean=0.985
  ALLOCARIDARA_ATTACK: n=100, acc=100.00%, conf_mean=0.992
  HEALTHY_LEAF: n=100, acc=100.00%, conf_mean=0.996
  LEAF_BLIGHT: n=100, acc=98.00%, conf_mean=0.978
  PHOMOPSIS_LEAF_SPOT: n=100, acc=100.00%, conf_mean=0.981

Summary saved: E:\Master\thesis_durian\models\classification\gradcam\gradcam_summary.json
